In [0]:
import yaml
from databricks.sdk import WorkspaceClient
from databricks.labs.dqx.profiler.generator import DQGenerator
from pyspark.sql import functions as F

ws = WorkspaceClient()

file_path = "/Volumes/places/enterprise_steady_state/ess0_linz_json/linz_data_contract/test2.yml"

with open(file_path, "r") as f:
    contract = yaml.safe_load(f)

# Generate DQX rules (including AI-assisted text rules)
generator = DQGenerator(workspace_client=ws)
rules = generator.generate_rules_from_contract(
    contract_file=file_path,
    process_text_rules=False,  # Disable AI-assisted rule generation to avoid endpoint requirement
    default_criticality="error"  # or "warn" for warnings instead of errors
)

#print(f"Generated {len(rules)} quality rules from contract")

# 1. Load the raw JSON file
raw_df = spark.read.option("multiLine", "true").json("/Volumes/places/enterprise_steady_state/ess0_linz_json/linz_property_unit_of_property_changeset/property_unit_of_property_changeset_5_2026-05-04_21-11-55.json")

# 2. Flatten the 'features' array so each feature is a row
# This extracts the 'properties', 'geometry', and 'id' into top-level columns
df = raw_df.select(F.explode("features").alias("feature")) \
    .select(
        "feature.id",
        "feature.geometry.type",
        "feature.geometry.coordinates",
        "feature.properties.*" # Flattens all keys inside 'properties'
    )

display(df)

In [0]:
import yaml
from pyspark.sql import functions as F
from pyspark.sql.types import *

def get_dynamic_projections(yaml_path):
    """
    Parses the YAML and returns a list of Spark Column expressions 
    mapping sourcePath to name with correct type casting.
    """
    # Type mapping for casting
    TYPE_MAPPING = {
        "string": StringType(),
        "double": DoubleType(),
        "float": FloatType(),
        "integer": IntegerType(),
        "long": LongType(),
        "boolean": BooleanType()
    }

    with open(yaml_path, 'r') as file:
        config = yaml.safe_load(file)

    properties = config.get('schema', [{}])[0].get('properties', [])
    
    expressions = []
    for prop in properties:
        name = prop.get('name')
        source_path = prop.get('sourcePath')
        p_type = prop.get('physicalType', 'string').lower()
        spark_type = TYPE_MAPPING.get(p_type, StringType())

        if source_path:
            # Create a column expression from the source path and cast it
            expr = F.col(source_path).cast(spark_type).alias(name)
        else:
            # Handle null sourcePaths by creating a null literal of the correct type
            expr = F.lit(None).cast(spark_type).alias(name)
        
        expressions.append(expr)
        
    return expressions

# 1. Generate the expressions from your YAML
yaml_file = "/Volumes/places/enterprise_steady_state/ess0_linz_json/linz_data_contract/linz_data_contract_with_sourcepath.yml"
column_projections = get_dynamic_projections(yaml_file)

# 2. Build the streaming DataFrame
df_raw = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", "/Volumes/places/enterprise_steady_state/ess0_linz_json/_schema/linz_property_unit_of_property_changeset")
    .option("multiLine", "true")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .load("/Volumes/places/enterprise_steady_state/ess0_linz_json/linz_property_unit_of_property_changeset/")
)

# 3. Apply the transformations: Explode and then Project
df = (
    df_raw
    .withColumn("feature", F.explode("features"))
    .select(*column_projections)
)
# Add processing timestamp for CDC sequencing
df.append(F.current_timestamp().alias("processing_timestamp"))
